In [ ]:
import geopandas as gpd
from pathlib import Path
import fiona
import matplotlib.pyplot as plt

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"
# Define the output directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
# Path to your rail GeoPackage
rail_gpkg_path = networks_folder / "transport/rail.gpkg"
layers = fiona.listlayers(rail_gpkg_path)
print("Available layers:", layers)

In [ ]:
# Read the roads layers (edges and nodes) from the GeoPackage.
rail_edges = gpd.read_file(rail_gpkg_path, layer="edges")
rail_nodes = gpd.read_file(rail_gpkg_path, layer="nodes")

rail_edges = rail_edges.to_crs(jamaica_metric_grid_crs)
rail_nodes = rail_nodes.to_crs(jamaica_metric_grid_crs)

In [ ]:
rail_edges.columns

In [ ]:
rail_nodes.columns

In [ ]:
# === For Line Features (Rail Edges) ===
# Perform an overlay (intersection) between the road edges and hydrobasins.
# This will split the roads by the hydrobasin boundaries.
rail_edges_overlay = gpd.overlay(rail_edges, hydrobasins, how="intersection")

# Optionally, calculate the length of each road segment (assuming a projected CRS)
rail_edges_overlay["length"] = rail_edges_overlay.geometry.length

# Example aggregation: Sum of road lengths by hydrobasin catchment (using HYBAS_ID)
rail_length_by_catchment = rail_edges_overlay.groupby("HYBAS_ID")["length"].sum().reset_index()
print("Rail Length by Catchment:")
print(rail_length_by_catchment)

In [ ]:
# === For Point Features (Road Nodes) ===
# Perform a spatial join to attach hydrobasin attributes (e.g., HYBAS_ID) to each node.
rail_nodes_join = gpd.sjoin(rail_nodes, hydrobasins, how="left", predicate="intersects")

display(rail_nodes_join.head())
display(rail_nodes_join.columns)

# Example aggregation: Count the number of nodes per catchment
nodes_count_by_catchment = rail_nodes_join.groupby("HYBAS_ID").size().reset_index(name="node_count")
display("\nNode Count by Catchment:")
display(nodes_count_by_catchment)

In [ ]:
# Define the output file path for the road edges layer
rail_edges_catchments_intersection = networks_catchments_intersections / "rail_edges_catchments_intersection.gpkg"

# Save the intersected edges layer to the new GeoPackage file
rail_edges_overlay.to_file(rail_edges_catchments_intersection, layer="intersected_edges", driver="GPKG")

# Save the nodes join layer to a new GeoPackage
rail_nodes_catchments_intersection = networks_catchments_intersections / "rail_nodes_catchments_intersection.gpkg"
rail_nodes_join.to_file(rail_nodes_catchments_intersection, layer="joined_nodes", driver="GPKG")